In [9]:
import numpy as np
from sklearn import preprocessing

In [10]:
raw_csv_data=np.loadtxt("Audiobooks_data.csv",delimiter=",")
unscaled_inputs_all=raw_csv_data[:,1:-1]
targets_all = raw_csv_data[:, -1].astype(int).flatten()


**BALANCE THE DATASET**

In [11]:
# Count how many targets are 1
num_one_targets = int(np.sum(targets_all))
zero_targets_counter = 0
indices_to_remove = []

# Balance the data: keep only as many 0s as there are 1s
for i in range(targets_all.shape[0]):
    if targets_all[i] == 0:
        zero_targets_counter += 1
        if zero_targets_counter > num_one_targets:
            indices_to_remove.append(i)

# Remove excess 0s
unscaled_inputs_equal_priors = np.delete(unscaled_inputs_all, indices_to_remove, axis=0)
targets_equal_priors = np.delete(targets_all, indices_to_remove, axis=0)

# Scale the inputs
scaled_inputs = preprocessing.scale(unscaled_inputs_equal_priors)

# Shuffle the data
shuffled_indices = np.arange(scaled_inputs.shape[0])
np.random.shuffle(shuffled_indices)

shuffled_inputs = scaled_inputs[shuffled_indices]
shuffled_targets = targets_equal_priors[shuffled_indices]

# Split into train, validation, and test sets
sample_size = shuffled_inputs.shape[0]

train_sample_count = int(0.8 * sample_size)
validation_sample_count = int(0.1 * sample_size)
test_sample_count = sample_size - train_sample_count - validation_sample_count

train_inputs = shuffled_inputs[:train_sample_count]
train_targets = shuffled_targets[:train_sample_count]

validation_inputs = shuffled_inputs[train_sample_count:train_sample_count + validation_sample_count]
validation_targets = shuffled_targets[train_sample_count:train_sample_count + validation_sample_count]

test_inputs = shuffled_inputs[train_sample_count + validation_sample_count:]
test_targets = shuffled_targets[train_sample_count + validation_sample_count:]

# Save the datasets to .npz files
np.savez("Audiobooks_data_train", inputs=train_inputs, targets=train_targets)
np.savez("Audiobooks_data_validation", inputs=validation_inputs, targets=validation_targets)
np.savez("Audiobooks_data_test", inputs=test_inputs, targets=test_targets)

In [12]:
import tensorflow as tf

C:\Users\Hammad\anaconda3\envs\tf_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [18]:
# Load training data
npz = np.load("Audiobooks_data_train.npz")
train_inputs = npz["inputs"].astype(np.float32)
train_targets = npz["targets"].astype(np.int32)

# Load validation data
npz = np.load("Audiobooks_data_validation.npz")
validation_inputs = npz["inputs"].astype(np.float32)
validation_targets = npz["targets"].astype(np.int32)

# Load test data
npz = np.load("Audiobooks_data_test.npz")
test_inputs = npz["inputs"].astype(np.float32)
test_targets = npz["targets"].astype(np.int32)

# Model architecture
input_size = train_inputs.shape[1]  # Automatically get input size
output_size = 2
hidden_layer_size = 50

model = tf.keras.Sequential([
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(output_size, activation='softmax')
])

# Compile model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train model
NUM_EPOCHS = 100
BATCH_SIZE = 100  # You can tune this later
early_stopping=tf.keras.callbacks.EarlyStopping(patience=2)

model.fit(train_inputs,
          train_targets,
          epochs=NUM_EPOCHS,
          batch_size=BATCH_SIZE,
          callbacks=[early_stopping],
          validation_data=(validation_inputs, validation_targets),
          verbose=2)

test_loss, test_accuracy = model.evaluate(test_inputs, test_targets)
print(f"Test accuracy: {test_accuracy:.2f}")

Epoch 1/100
36/36 - 2s - 45ms/step - accuracy: 0.6778 - loss: 0.6050 - val_accuracy: 0.7875 - val_loss: 0.4994
Epoch 2/100
36/36 - 0s - 8ms/step - accuracy: 0.7798 - loss: 0.4558 - val_accuracy: 0.8210 - val_loss: 0.4114
Epoch 3/100
36/36 - 0s - 10ms/step - accuracy: 0.7946 - loss: 0.4039 - val_accuracy: 0.8233 - val_loss: 0.3778
Epoch 4/100
36/36 - 0s - 8ms/step - accuracy: 0.7969 - loss: 0.3815 - val_accuracy: 0.8322 - val_loss: 0.3610
Epoch 5/100
36/36 - 0s - 7ms/step - accuracy: 0.8072 - loss: 0.3678 - val_accuracy: 0.8054 - val_loss: 0.3615
Epoch 6/100
36/36 - 0s - 8ms/step - accuracy: 0.8039 - loss: 0.3612 - val_accuracy: 0.8434 - val_loss: 0.3388
Epoch 7/100
36/36 - 0s - 7ms/step - accuracy: 0.8072 - loss: 0.3518 - val_accuracy: 0.8434 - val_loss: 0.3355
Epoch 8/100
36/36 - 0s - 9ms/step - accuracy: 0.8050 - loss: 0.3505 - val_accuracy: 0.8412 - val_loss: 0.3272
Epoch 9/100
36/36 - 0s - 10ms/step - accuracy: 0.8142 - loss: 0.3458 - val_accuracy: 0.8456 - val_loss: 0.3352
Epoch 1